In [14]:
import os
import requests
import logging
import asyncio
import torch
from typing import Literal
from pydantic import BaseModel, Field
import operator
import json
import re
from dotenv import load_dotenv
import pickle
import bm25s

# LangChain / LangGraph Imports
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage,AIMessage
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages.utils import trim_messages
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.retrievers import BaseRetriever
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command
from langgraph.managed.is_last_step import RemainingSteps
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
import yfinance as yf
from typing import List, Literal


In [ ]:
# 1. Observability and Logging Setup

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class LangchainBM25sRetriever(BaseRetriever):
    """Custom LangChain retriever wrapper around the high-speed bm25s library."""
    documents : list[Document]
    retriever_index: bm25s.BM25
    k: int = 4

    class Config:
        arbitrary_types_allowed = True
    
    @classmethod
    def from_documents(cls,documents:List[Document], k:int = 4, method: str="atire"):
        if not documents:
            raise ValueError("Document list passed to BM25s index is empty.")
        
        # Extract corpus strings and build vocabulary matrix
        corpus = [doc.page_content for doc in documents]
        corpus_tokens = bm25s.tokenize(corpus, stopwords="en")
        
        # Instantiate and build index via backend sparse arrays
        retriever_index = bm25s.BM25(method=method)
        retriever_index.index(corpus_tokens)
        
        return cls(documents=documents, retriever_index=retriever_index, k=k)
    
    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None) -> List[Document]:
        """Synchronous query implementation mapping predictions back to raw Docs."""
        query_tokens = bm25s.tokenize(query, stopwords="en")
        results, scores = self.retriever_index.retrieve(
            query_tokens, k=self.k, show_progress=False
        )
        
        retrieved_docs = []
        # Safely loop through top indices extracted from matrix dimension
        for idx, score in zip(results[0], scores[0]):
            doc = self.documents[idx]
            doc.metadata["bm25_score"] = float(score)
            retrieved_docs.append(doc)
            
        return retrieved_docs


#2. Database Connection and Index Compilation

logger.info("Connecting to local Chroma Vector DB...")
device ="mps" if torch.backends.mps.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': device,
                  'trust_remote_code': True},
    encode_kwargs={'normalize_embeddings': True}
)
#Establishing connection to DB.
vector_db = Chroma(
    persist_directory="./penny_stock_vector_db",
    embedding_function=embeddings,
    collection_name="langchain_large"
)
dense_retriever = vector_db.as_retriever(search_type="mmr", search_kwargs={"k": 6, "fetch_k": 30})

#Sparse Retriever (Exact Keywords)
logger.info("Building BM25 Index from existing Vector DB data...")
raw_data = vector_db.get() #Pulls all documents currently in the DB

# Convert raw DB strings back into Document objects for BM25
documents = [
    Document(page_content=text, metadata=meta)
    for text, meta in zip(raw_data['documents'], raw_data['metadatas'])
]

bm25s_retriever = LangchainBM25sRetriever.from_documents(documents,k=4)


# ==========================================
# Custom Reciprocal Rank Fusion Engine
# ==========================================
async def custom_ensemble_retrieve(query: str, dense_retriever, sparse_retriever, k=4, c=60):
    """
    Manually fuses results from a dense and sparse retriever using Reciprocal Rank Fusion.
    Bypasses the need for LangChain's EnsembleRetriever import entirely.
    """
    logger.info(f"Running Custom Hybrid Search for: {query}")
    
    # 1. Get results from BOTH retrievers independently
    dense_docs, sparse_docs = await asyncio.gather(
        dense_retriever.ainvoke(query),
        asyncio.to_thread(sparse_retriever.invoke, query)
    )
    
    # 2. Setup a dictionary to hold our fused mathematical scores
    fused_scores = {}
    
    # 3. Score the Vector DB results
    for rank, doc in enumerate(dense_docs):
        doc_content = doc.page_content
        if doc_content not in fused_scores:
            fused_scores[doc_content] = {"doc": doc, "score": 0}
        fused_scores[doc_content]["score"] += 1 / (rank + c)
        
    # 4. Score the BM25 Keyword results
    for rank, doc in enumerate(sparse_docs):
        doc_content = doc.page_content
        if doc_content not in fused_scores:
            fused_scores[doc_content] = {"doc": doc, "score": 0}
        fused_scores[doc_content]["score"] += 1 / (rank + c)

    # 5. Sort all documents by their new fused score (highest to lowest)
    reranked_results = sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in reranked_results[:k]]

#3. Tool Definitions

@tool
async def get_indian_stock_data(symbol: str, exchange: str = "NSE") -> str:
    """"This Function fetches the real-time and historical stock data for Indian equities from NSE or BSE"""
    try:
        #Format the ticker symbol for Yahoo Finance
        exchange_suffix = ".NS" if exchange.upper() == "NSE" else ".BO"
        ticker_symbol = f"{symbol.upper()}{exchange_suffix}"

        logger.info("Fetching the live data for: {ticker_symbol}")
        ticker = yf.Ticker(ticker_symbol)

        # Grab the live dictionary of stock information
        info = ticker.info

        # Check if the stock actually exists (yfinance returns a specific dict if not found)
        if "currentPrice" not in info and "regularMarketPrice" not in info:
            return f"Error: Could not find live data for {ticker_symbol}. The symbol might be incorrect."
        
        clean_data = {
            "Symbol": info.get("symbol"),
            "Current Price": info.get("currentPrice", info.get("regularMarketPrice")),
            "Previous Close": info.get("previousClose"),
            "Day High": info.get("dayHigh"),
            "Day Low": info.get("dayLow"),
            "Volume": info.get("volume"),
            "Market Cap": info.get("marketCap")
        }

        return str(clean_data)
    
    except Exception as e:
        logger.warning(f"yfinance API failure for {symbol}: {e}")
        return f"Error fetching data for {symbol}." 

@tool
async def penny_stock_researcher(query: str) -> str:
    """Searches the local vector database for SEC filings, news, and historical context."""
    logger.info(f"Querying Vector DB for: {query}")
    docs = await custom_ensemble_retrieve(query, dense_retriever, bm25s_retriever)
    return "\n\n".join([doc.page_content for doc in docs])


# We keep the tools separate so we can assign specific tools to specific agents
data_tools = [get_indian_stock_data]
analyst_tools = [penny_stock_researcher]

#4. GPU-Accelerated LLM Configuration

llm_data_agent = ChatOllama(model="penny-llama", temperature=0.0, keep_alive=-1,num_ctx=8192)
llm_analyst_agent = ChatOllama(model="qwen3.5:35b", temperature=0.2, keep_alive=0, num_ctx=8192)
llm_judge_agent = ChatOllama(model="ggozad/prometheus2:latest", temperature=0.0, keep_alive=0, num_ctx=8192)

class AgentState(MessagesState):
    remaining_steps: RemainingSteps

#5. Async Multi-Agent Nodes

async def supervisor_node(state: AgentState) -> Command:
    # --- DETERMINISTIC GOVERNOR ---
    if state.get("remaining_steps", 25) <= 2:
        logger.warning("Recursion limit approaching. Forcefully terminating graph.")
        return Command(goto=END)

    #Simple context truncation using a string length metric helper
    messages_to_inspect = state["messages"]
    sanitized_messages = []
    for msg in messages_to_inspect:
        if isinstance(msg, ToolMessage) and len(str(msg.content)) > 1200:
            truncated_content = str(msg.content)[:1200] + "..."
            sanitized_messages.append(ToolMessage(content=truncated_content, tool_call_id=msg.tool_call_id))
        else:
            sanitized_messages.append(msg)

    system_prompt = SystemMessage(
        content=(
            "You are the Financial Supervisor orchestrating a strict sequential pipeline.\n"
            "ROUTING RULES:\n"
            "1. If the chat history is missing the raw live stock data, route to 'MarketDataAgent'.\n"
            "2. If the raw data is present, but there is NO comprehensive synthesis report from the 'FinancialAnalystAgent', route to 'FinancialAnalystAgent'.\n"
            "3. NEVER route the same task to the same agent more than once sequentially.\n"
            "4. If the 'FinancialAnalystAgent' has generated the final analytical report, you MUST route to 'EvaluatorAgent' for quality assessment. Do NOT route to 'FINISH' yourself.\n\n"
            "--- STRICT OUTPUT FORMAT ---\n"
            "You MUST respond ONLY with a valid JSON object. Do NOT include any conversational text, explanations, or thoughts (e.g., 'Let me try...').\n"
            "Your output must be pure JSON."
        )
    )

    class Router(BaseModel):
        next_agent: Literal["MarketDataAgent", "FinancialAnalystAgent", "EvaluatorAgent", "FINISH"] = Field(
            description="The next agent to route to based on the current context."
        )
    
    router = llm_data_agent.with_structured_output(Router, method="json_schema")

    try:
        decision = await router.ainvoke([system_prompt] + sanitized_messages)
    except Exception as e:
        logger.error(f"Supervisor failed to output valid structured JSON: {e}. Defaulting to FINISH.")
        return Command(goto=END)

    logger.info(f"Supervisor routing to: {decision.next_agent}")

    if decision.next_agent == "FINISH":
        return Command(goto=END)
    return Command(goto=decision.next_agent)
    
async def market_data_agent(state: AgentState) -> Command[Literal["tools", "supervisor_node"]]:
    system_prompt = SystemMessage(
        content=(
            "You are a strictly constrained Market Data Agent. "
            "Your ONLY job is to fetch NSE/BSE data using the provided tools. "
            "CRITICAL: DO NOT analyze the data, do not compare it to PDFs, and do not write a report. "
            "If the tool returns an error, output the error message so the user knows it failed. "
            "Only if the tool returns actual JSON data, reply with: 'Data fetched successfully. Ready for the FinancialAnalystAgent to perform the analysis.'"
        )
    )
    

    response = await llm_data_agent.bind_tools(data_tools).ainvoke([system_prompt] + state["messages"])

    if hasattr(response, "tool_calls") and response.tool_calls:
        return Command(goto="tools", update={"messages": [response]})
    return Command(goto="supervisor_node", update={"messages": [response]})

async def financial_analyst_agent(state: AgentState) -> Command[Literal["tools", "supervisor_node"]]:
    system_prompt = SystemMessage(
        content=(
            "You are a Senior Financial Analyst. Use the penny_stock_researcher tool to search the local database "
            "for risk context. Synthesize live data and database context into a professional assessment."
        )
    )
    
    #To check if the message contains the feedback from Evaluator LLM
    last_msg_content = str(state["messages"][-1].content)
    is_feedback = "Your previous report failed quality checks" in last_msg_content

    if is_feedback:
        payload = [system_prompt] + state["messages"]
    else:
        transitional_msg = HumanMessage(
            content="The tool execution is complete. Please review the data above and proceed with your analysis or further tool calls."
        )
        payload = [system_prompt] + state["messages"] + [transitional_msg]

    # Inject the transitional message at the very end of the payload passed to the LLM
    response = await llm_analyst_agent.bind_tools(analyst_tools).ainvoke(payload)
    
    if hasattr(response, "tool_calls") and response.tool_calls:
        return Command(goto="tools", update={"messages": [response]})
    return Command(goto="supervisor_node", update={"messages": [response]})

async def evaluator_node(state: AgentState) -> Command[Literal["FinancialAnalystAgent","__end__"]]:
    """Prometheus-2 Judgement Node. Evaluates the Analyst's report for faithfulness and relevance."""
    logger.info("Evaluating Analyst Report using Prometheus-2...")


    #Extract the last message (the generated content)
    report_content = state["messages"][-1].content

    instruction = "Evaluate the financial report generated by the Analyst based on the provided live data and vector context."
    reference_answer = "A comprehensive financial report that perfectly synthesizes live market data and database context without hallucination, strictly adhering to the provided facts."
    score_rubric = "Score 1 to 5 based on Faithfulness and Relevance. A score of 5 indicates the report relies solely on the provided SEC vector filings without injecting external bias, and directly addresses the prompt. A score of 1 indicates severe hallucination or irrelevance."
    
    # Prometheus-2 exact prompt template structure
    prompt = f"""###Task Description:
An instruction (might include an Input inside it), a response to evaluate, a reference answer that gets a score of 5, and a score rubric representing a evaluation criteria are given.
1. Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general. Refer to the given context when writing the feedback and making an assessment.
2. After writing a feedback, write a score that is an integer between 1 and 5. You should refer to the score rubric.
3. The output format should look as follows: "(write a feedback for criteria) (an integer number between 1 and 5)"
4. Please do not generate any other opening, closing, and explanations.

###The instruction to evaluate:
{instruction}

###Response to evaluate:
{report_content}

###Reference Answer (Score 5):
{reference_answer}

###Score Rubrics:
{score_rubric}

###Feedback:
"""
    response = await llm_judge_agent.ainvoke([HumanMessage(content=prompt)])
    eval_text = response.content
    logger.info(f"Prometheus Evaluation Output: {eval_text}")

    score = 5 #default saftey variable
    match = re.search(r'(\d)', eval_text)
    if match:
        score = int(match.group(1))

    logger.info(f"Assigned Quality Score: {score}")

    # LangGraph routing logic using the Command handoff pattern
    if score >= 4:
        logger.info("Report passed evaluation. Ending workflow.")
        # Append a final AI Message so that print loop displays the Judge's score
        final_msg = AIMessage(content=f"LLM Judge Evaluation Passed (Score:{score})\n\nFeedback: {eval_text}")
        return Command(goto=END, update={"messages": [final_msg]})
    else:
        logger.warning("Report failed evaluation. Routing back to FinancialAnalystAgent for revision.")
        # Append the judge's feedback so the analyst can self-correct in the next cycle
        feedback_msg = HumanMessage(
            content=f"Your previous report failed quality checks. Here is the evaluator's feedback:\n{eval_text}\n\nPlease revise the report to strictly adhere to the data and eliminate hallucinations."
        )
        return Command(goto="FinancialAnalystAgent", update={"messages": [feedback_msg]})




async def execute_tools(state: AgentState) -> Command:
    """Smart tool router: executes the tool and returns to the agent that called it."""
    last_msg = state["messages"][-1]
    tool_msgs = []
    return_target = "supervisor_node" # Fallback

    for tc in last_msg.tool_calls:
        args = tc["args"]
        
        if tc["name"] == "get_indian_stock_data":
            # --- QWEN 3.5 OPTIMIZED SANITIZATION ---
            # Qwen typically outputs clean strings, but we keep a lightweight fallback
            exchange = args.get("exchange", "NSE")
            symbol = args.get("symbol", "")

            # If it occasionally nests the value, safely extract it
            if isinstance(exchange, dict): exchange = exchange.get("value", "NSE")
            if isinstance(symbol, dict): symbol = symbol.get("value", "")
            elif isinstance(symbol, list): symbol = symbol[-1]

            # Enforce strict string formatting for the API
            args["exchange"] = str(exchange).upper()
            args["symbol"] = str(symbol).upper()
            # ---------------------------------------
                 
            result = await get_indian_stock_data.ainvoke(args)
            return_target = "MarketDataAgent"

        elif tc["name"] == "penny_stock_researcher":
            result = await penny_stock_researcher.ainvoke(tc["args"])
            return_target = "FinancialAnalystAgent"
            
        else:
            # --- HALLUCINATION FALLBACK ---
            logger.warning(f"LLM hallucinated an unknown tool: {tc['name']}")
            result = f"System Error: The tool '{tc['name']}' does not exist. Please use the correct provided tool."
            # The return_target defaults to supervisor_node as defined at the top
            return_target = "FinancialAnalystAgent"
        
        tool_msgs.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
        logger.info(f"Tool executed: {tc['name']}")
    
    return Command(goto=return_target, update={"messages": tool_msgs})

#6. Graph Compilation

builder = StateGraph(AgentState)

builder.add_node("supervisor_node", supervisor_node)
builder.add_node("MarketDataAgent", market_data_agent)
builder.add_node("FinancialAnalystAgent", financial_analyst_agent)
builder.add_node("EvaluatorAgent", evaluator_node)
builder.add_node("tools", execute_tools)

builder.add_edge(START, "supervisor_node")

#7. Async Execution Loop

async def main():
    # Step 1: We bypass the Postgres checkpointer entirely to isolate the graph logic
    app = builder.compile()
    
    thread_config = {"configurable": {"thread_id": "audit_trace_txn_137"}}
    user_query = "Fetch the live data for Suzlon on the BSE. Then check the PDF documents to see if buying penny stocks is safe."
    
    print(f"\n[User Query]: {user_query}\n")
    
    # Step 2: We REMOVED the try/except block. 
    # If the code fails now, Python will give us the raw stack trace and the exact line number!
    async for event in app.astream({"messages": [HumanMessage(content=user_query)]}, config=thread_config):
        for node_name, node_state in event.items():
            print(f"\n--- Output from {node_name} ---")
            if node_state and "messages" in node_state and node_state["messages"]:
                msg = node_state["messages"][-1]
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    print(f"Action: Preparing to call tool -> {[t['name'] for t in msg.tool_calls]}")
                elif msg.content:
                    print(msg.content)

await main()

/var/folders/9l/yjhwmfrs4m379sjy1069t9lm0000gn/T/ipykernel_78500/1529060397.py:6: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class LangchainBM25sRetriever(BaseRetriever):
2026-05-24 15:59:38,020 - INFO - Connecting to local Chroma Vector DB...
2026-05-24 15:59:38,268 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-24 15:59:38,280 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
2026-05-24 15:59:38,501 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-24 15:59:38,514 - INFO - HTTP Request: HEAD https://huggingfa

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-05-24 15:59:40,379 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3 "HTTP/1.1 200 OK"
2026-05-24 15:59:40,478 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-24 15:59:40,642 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/main "HTTP/1.1 200 OK"
2026-05-24 15:59:40,702 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-24 15:59:40,897 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/discussions?p=0 "HTTP/1.1 200 OK"
2026-05-24 15:59:40,928 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-24 15:59:41,229 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/refs%2Fpr%2F130 "HTTP/1.1 200 OK"
2026-05-24 15:59:41,230 - INFO - H

Split strings:   0%|          | 0/618 [00:00<?, ?it/s]

2026-05-24 15:59:45,836 - DEBUG - Building index from IDs objects


...


--- Output from FinancialAnalystAgent ---
# Professional Financial Assessment: Suzlon Energy & Penny Stock Risk Analysis

## Executive Summary

Based on live BSE data and historical risk context, I've conducted a comprehensive analysis of Suzlon Energy and penny stock safety considerations.

---

## 1. Suzlon Energy (SUZLON.BO) - Current Status

| Metric | Value |
|--------|-------|
| **Current Price** | ₹53.78 |
| **Previous Close** | ₹52.89 |
| **Day High** | ₹53.96 |
| **Day Low** | ₹53.01 |
| **Volume** | 5.90M shares |
| **Market Cap** | ₹73,763 crore |

### Key Observations:
- **NOT a Penny Stock**: At ₹53.78 per share, Suzlon is classified as a **large-cap stock**, not a penny stock
- **Market Cap**: ₹73,763 crore places it firmly in large-cap territory
- **Trading Activity**: Healthy daily volume of ~5.9M shares indicates good liquidity

---

## 2. Penny Stock Safety Analysis

### ⚠️ Critical Risk Findings:

| Risk Factor | Assessment |
|-------------|------------|
| **Price M

2026-05-24 16:02:28,289 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-05-24 16:02:28,468 - INFO - Supervisor routing to: EvaluatorAgent
2026-05-24 16:02:28,471 - INFO - Evaluating Analyst Report using Prometheus-2...



--- Output from supervisor_node ---


2026-05-24 16:02:32,334 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-05-24 16:02:37,866 - INFO - Prometheus Evaluation Output: 
The response demonstrates a clear understanding of the task at hand, providing an in-depth analysis of Suzlon Energy's financial status and penny stock safety considerations. It effectively utilizes live BSE data to assess the current price, previous close, day high, day low, volume, and market capitalization of Suzlon Energy. The response also provides a detailed risk assessment for penny stocks, highlighting critical factors such as price misconception, historical precedents, data scarcity, volatility risk, and liquidity concerns.

Furthermore, the response offers actionable investment recommendations for both Suzlon Energy and penny stock investments, emphasizing the importance of due diligence and diversification. The conclusion effectively summarizes the key findings and provides a clear recommendation on how to appro


--- Output from EvaluatorAgent ---
LLM Judge Evaluation Passed (Score:5)

Feedback: 
The response demonstrates a clear understanding of the task at hand, providing an in-depth analysis of Suzlon Energy's financial status and penny stock safety considerations. It effectively utilizes live BSE data to assess the current price, previous close, day high, day low, volume, and market capitalization of Suzlon Energy. The response also provides a detailed risk assessment for penny stocks, highlighting critical factors such as price misconception, historical precedents, data scarcity, volatility risk, and liquidity concerns.

Furthermore, the response offers actionable investment recommendations for both Suzlon Energy and penny stock investments, emphasizing the importance of due diligence and diversification. The conclusion effectively summarizes the key findings and provides a clear recommendation on how to approach investing in large-cap versus penny stocks.

The response is well-structured

In [2]:
import json
from huggingface_hub import hf_hub_download

src = json.load(open(hf_hub_download("unsloth/Meta-Llama-3.1-8B-Instruct", "tokenizer_config.json")))["chat_template"]
out = "/Users/saketh/models/Llama-3-8B-mlx-q4/chat_template.jinja"
open(out, "w").write(src)
print("Wrote", len(src), "chars. Knows tools:", "tool" in src.lower())

Wrote 4614 chars. Knows tools: True


In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("/Users/saketh/models/Llama-3-8B-mlx-q4")
d = json.loads(open("/Users/saketh/PennyStockLLM/finetune/data/train.jsonl").readline())
print(tok.apply_chat_template(d["messages"], tools=d["tools"], tokenize=False))]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a strictly constrained Market Data Agent. Your ONLY job is to fetch raw live stock data for Indian equities using the available tool. Always call the tool; never answer from memory.<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}.Do not use variables.

{
    "type": "function",
    "function": {
        "name": "get_indian_stock_data",
        "description": "Fetches the real-time and historical stock data for Indian equities from NSE or BSE.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
   